In [ ]:
import subprocess
import sys

def install_if_missing(package):
    """Check if package is installed, if not install it."""
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

install_if_missing('pandas')
install_if_missing('numpy')
install_if_missing('matplotlib')
install_if_missing('scipy')
install_if_missing('astropy')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
from scipy.interpolate import interp1d
from astropy.coordinates import get_body_barycentric_posvel, GCRS, SkyCoord, ICRS, Galactic, GeocentricMeanEcliptic
from astropy.time import Time
import astropy.units as u

In [ ]:
# Physical constants
G        = 6.673e-11   # N m^2 / kg^2
M_Earth  = 6.0e24      # kg
M_Sun    = 1.989e30    # kg
M_Moon   = 7.342e22    # kg
R_earth  = 6371e3      # m
mu       = G * M_Earth # gravitational parameter [m^3/s^2]

# Orbit geometry
R_orbit  = 91e6 + R_earth          # semi-major axis [m]
orbit_inclination = 5 * math.pi / 180  # rad

# Formation parameters
d_spacecraft = 68.84     # inter-spacecraft separation [m]
M_oculus     = 1500    # kg
M_lux        = 500     # kg

# Derived orbital quantities
v_spacecraft_0 = math.sqrt(G * M_Earth / R_orbit)  # circular velocity [m/s]
T_orbit        = 2 * math.pi * R_orbit / v_spacecraft_0

# Integration grid
N_steps = 30000
dt      = T_orbit / N_steps
t       = np.linspace(0, T_orbit, N_steps)

# J2 oblateness coefficient
J2 = 1.08263e-3

# Solar radiation pressure (at 1 AU)
P_solar = 4.56e-6  # N/m^2
AU      = 149597870700  # m
C_R     = 1.5          # reflectivity
A_sc    = 1.0          # cross-sectional area [m^2]
m_sc    = 500.0        # spacecraft mass for SRP [kg]

# Simulation epoch
t0_epoch = Time("2025-01-01T00:00:00", scale="utc")


In [ ]:
# Interpolated Sun/Moon ECI positions sampled every 100 steps for efficiency
def get_sun_moon_eci(t_sec):
    t = t0_epoch + t_sec * u.s
    sun_gcrs  = get_body_barycentric_posvel('sun',  t, ephemeris='builtin')[0]
    moon_gcrs = get_body_barycentric_posvel('moon', t, ephemeris='builtin')[0]
    def to_gcrs(coord):
        c = SkyCoord(coord, frame='icrs', obstime=t, representation_type='cartesian')
        c = c.transform_to(GCRS(obstime=t))
        return np.array([c.cartesian.x.to(u.m).value,
                         c.cartesian.y.to(u.m).value,
                         c.cartesian.z.to(u.m).value])
    return to_gcrs(sun_gcrs), to_gcrs(moon_gcrs)

print("Pre-computing Sun/Moon ephemeris table...")
sample_idx  = np.arange(0, N_steps, 100)
_sun_samp   = np.zeros((len(sample_idx), 3))
_moon_samp  = np.zeros((len(sample_idx), 3))
for k, idx in enumerate(sample_idx):
    _sun_samp[k], _moon_samp[k] = get_sun_moon_eci(t[idx])

_sun_interp  = interp1d(t[sample_idx], _sun_samp,  axis=0, fill_value="extrapolate")
_moon_interp = interp1d(t[sample_idx], _moon_samp, axis=0, fill_value="extrapolate")

def r_sun_at(t_sec):  return _sun_interp(t_sec)
def r_moon_at(t_sec): return _moon_interp(t_sec)
print("Ephemeris table ready.")


In [ ]:
# Unit-vector helpers and galactic/ecliptic frame vectors at epoch
def _unit(v):
    v = np.asarray(v, dtype=float)
    return v / np.linalg.norm(v)

def get_galactic_center_gcrs(epoch):
    gc = SkyCoord(l=0*u.deg, b=0*u.deg, frame=Galactic()).transform_to(ICRS())
    return _unit(gc.transform_to(GCRS(obstime=epoch)).cartesian.xyz.to_value(u.one))

def get_north_galactic_pole_gcrs(epoch):
    ngp = SkyCoord(l=0*u.deg, b=90*u.deg, frame=Galactic()).transform_to(ICRS())
    return _unit(ngp.transform_to(GCRS(obstime=epoch)).cartesian.xyz.to_value(u.one))

def get_north_ecliptic_pole_gcrs(epoch):
    nep = SkyCoord(lon=0*u.deg, lat=90*u.deg,
                   frame=GeocentricMeanEcliptic(equinox=epoch)).transform_to(ICRS())
    return _unit(nep.transform_to(GCRS(obstime=epoch)).cartesian.xyz.to_value(u.one))

r_gc_hat  = get_galactic_center_gcrs(t0_epoch)
r_ngp_hat = get_north_galactic_pole_gcrs(t0_epoch)
r_nep_hat = get_north_ecliptic_pole_gcrs(t0_epoch)


In [ ]:
# Perturbed acceleration: two-body + J2 + Sun/Moon third-body + SRP
def compute_perturbed_acceleration(position, t_sec=0.0):
    r_sun  = r_sun_at(t_sec)
    r_moon = r_moon_at(t_sec)
    r_norm = np.linalg.norm(position)

    # Two-body
    a_2body = -mu / r_norm**3 * position

    # J2 oblateness
    z2_r2 = (position[2] / r_norm) ** 2
    fJ2   = 1.5 * J2 * mu * R_earth**2 / r_norm**5
    a_J2  = fJ2 * np.array([
        position[0] * (5.0 * z2_r2 - 1.0),
        position[1] * (5.0 * z2_r2 - 1.0),
        position[2] * (5.0 * z2_r2 - 3.0)
    ])

    # Sun third-body
    d_sun  = r_sun - position
    a_sun  = G * M_Sun  * (d_sun  / np.linalg.norm(d_sun)**3
                           - r_sun  / np.linalg.norm(r_sun)**3)

    # Moon third-body
    d_moon = r_moon - position
    a_moon = G * M_Moon * (d_moon / np.linalg.norm(d_moon)**3
                           - r_moon / np.linalg.norm(r_moon)**3)

    # Solar radiation pressure
    sc_from_sun      = position - r_sun
    sc_from_sun_norm = np.linalg.norm(sc_from_sun)
    P_scaled         = P_solar * (AU / sc_from_sun_norm) ** 2
    a_srp            = (C_R * P_scaled * A_sc / m_sc) * (sc_from_sun / sc_from_sun_norm)

    return a_2body + a_J2 + a_sun + a_moon + a_srp, d_sun, d_moon


In [ ]:
# Point-mass body and GC-facing initial conditions
class Body:
    def __init__(self, mass, position, velocity):
        self.mass     = mass
        self.position = np.array(position, dtype=float)
        self.velocity = np.array(velocity, dtype=float)
    def copy(self):
        return Body(self.mass, self.position.copy(), self.velocity.copy())


def make_initial_conditions_face_gc(inclination, separation=d_spacecraft):
    r_oculus_0 = np.array([
        R_orbit * np.cos(inclination), 0.0, R_orbit * np.sin(inclination)
    ], dtype=float)
    v_oculus_0 = np.array([0.0, v_spacecraft_0, 0.0], dtype=float)

    rel_offset = separation * (r_gc_hat / np.linalg.norm(r_gc_hat))
    r_lux_0    = r_oculus_0 + rel_offset
    v_lux_0    = v_oculus_0.copy()

    orbital_normal = np.cross(r_oculus_0, v_oculus_0)
    orbital_normal = orbital_normal / np.linalg.norm(orbital_normal)

    return r_oculus_0, v_oculus_0, r_lux_0, v_lux_0, orbital_normal


In [ ]:
# RK4 integrator with rigid formation constraint (slave tracks master exactly)
class Simulation:
    def __init__(self, body1, body2, dt, nsteps, gravity_func):
        self.body1         = body1
        self.body2         = body2
        self.dt            = dt
        self.nsteps        = nsteps
        self.gravity_func  = gravity_func

        self.relative_offset = body2.position.copy() - body1.position.copy()

        self.r1            = np.zeros((nsteps, 3))
        self.v1            = np.zeros((nsteps, 3))
        self.r2            = np.zeros((nsteps, 3))
        self.v2            = np.zeros((nsteps, 3))
        self.f2_constraint = np.zeros((nsteps, 3))

        self.total_impulse = 0.0
        self.has_run       = False
        self._store_step(0)

    def _store_step(self, i):
        self.r1[i] = self.body1.position
        self.v1[i] = self.body1.velocity
        self.r2[i] = self.body2.position
        self.v2[i] = self.body2.velocity

    def _rk4_step_with_constraint(self, step):
        dt = self.dt
        d0 = self.relative_offset
        r1_0, v1_0 = self.body1.position.copy(), self.body1.velocity.copy()

        def acc(r_vec):
            a, _, _ = self.gravity_func(r_vec, step * dt)
            return a

        # RK4 stages (master)
        k1_r, k1_v = v1_0,                       acc(r1_0)
        k2_r, k2_v = v1_0 + 0.5*dt*k1_v,         acc(r1_0 + 0.5*dt*k1_r)
        k3_r, k3_v = v1_0 + 0.5*dt*k2_v,         acc(r1_0 + 0.5*dt*k2_r)
        k4_r, k4_v = v1_0 + dt*k3_v,             acc(r1_0 + dt*k3_r)

        r1_new = r1_0 + dt * (k1_r + 2*k2_r + 2*k3_r + k4_r) / 6.0
        v1_new = v1_0 + dt * (k1_v + 2*k2_v + 2*k3_v + k4_v) / 6.0

        # Constraint: slave required acceleration = master acceleration
        a_con_stages = [
            acc(r1_0)            - acc(r1_0 + d0),
            acc(r1_0+0.5*dt*k1_r) - acc(r1_0+0.5*dt*k1_r + d0),
            acc(r1_0+0.5*dt*k2_r) - acc(r1_0+0.5*dt*k2_r + d0),
            acc(r1_0+dt*k3_r)    - acc(r1_0+dt*k3_r + d0),
        ]
        a2_con_rk4 = (a_con_stages[0] + 2*a_con_stages[1]
                      + 2*a_con_stages[2] + a_con_stages[3]) / 6.0

        self.body1.position, self.body1.velocity = r1_new, v1_new
        self.body2.position = r1_new + d0
        self.body2.velocity = v1_new.copy()

        return a2_con_rk4

    def run(self):
        for i in range(1, self.nsteps):
            a2_con = self._rk4_step_with_constraint(i)
            self._store_step(i)
            self.f2_constraint[i] = self.body2.mass * a2_con
            self.total_impulse   += np.linalg.norm(self.f2_constraint[i]) * self.dt
        self.has_run = True


In [ ]:
# Initialise bodies and run one full orbital period
r_oc_0, v_oc_0, r_lx_0, v_lx_0, _ = make_initial_conditions_face_gc(orbit_inclination)
oculus = Body(M_oculus, r_oc_0, v_oc_0)
lux    = Body(M_lux,    r_lx_0, v_lx_0)

sim = Simulation(oculus, lux, dt, N_steps, compute_perturbed_acceleration)
sim.run()

delta_v = sim.total_impulse / M_lux
print(f"Total impulse on Lux : {sim.total_impulse:.4e} N·s")
print(f"Equivalent Δv        : {delta_v:.4f} m/s")


In [ ]:
# Constraint force magnitude over one orbit and cumulative Δv
t_hours = t / 3600
F_mag   = np.linalg.norm(sim.f2_constraint, axis=1)
dv_cum  = np.cumsum(F_mag * dt) / M_lux

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

axes[0].plot(t_hours, F_mag * 1e6, color='steelblue')
axes[0].set_ylabel("Constraint Force (μN)")
axes[0].set_title("Formation-Keeping Constraint Force on Lux")
axes[0].grid(True)

axes[1].plot(t_hours, dv_cum, color='darkorange')
axes[1].set_xlabel("Time (hours)")
axes[1].set_ylabel("Cumulative Δv (m/s)")
axes[1].set_title("Cumulative Δv Requirement")
axes[1].grid(True)

plt.tight_layout()
plt.savefig("constraint_force_dv.png", dpi=150)
plt.show()


In [ ]:
# 3-D trajectories of Oculus and Lux over one orbit
fig = plt.figure(figsize=(9, 9))
ax  = fig.add_subplot(111, projection='3d')

ax.plot(sim.r1[:, 0], sim.r1[:, 1], sim.r1[:, 2], label='Oculus', lw=0.8)
ax.plot(sim.r2[:, 0], sim.r2[:, 1], sim.r2[:, 2], label='Lux',    lw=0.8, ls='--')
ax.scatter(0, 0, 0, color='royalblue', s=120, label='Earth')

ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_zlabel("Z (m)")
ax.set_title("Spacecraft Trajectories – One Orbital Period")
ax.legend()
plt.tight_layout()
plt.savefig("trajectories_3d.png", dpi=150)
plt.show()


In [ ]:
# Per-perturbation force contributions evaluated along the Oculus trajectory
a_J2_arr  = np.zeros((N_steps, 3))
a_sun_arr = np.zeros((N_steps, 3))
a_moon_arr= np.zeros((N_steps, 3))
a_srp_arr = np.zeros((N_steps, 3))

for i in range(N_steps):
    pos   = sim.r1[i]
    r_sun = r_sun_at(t[i]);  r_moon = r_moon_at(t[i])
    r_n   = np.linalg.norm(pos)
    z2_r2 = (pos[2] / r_n)**2
    fJ2   = 1.5 * J2 * mu * R_earth**2 / r_n**5
    a_J2_arr[i] = fJ2 * np.array([pos[0]*(5*z2_r2-1), pos[1]*(5*z2_r2-1), pos[2]*(5*z2_r2-3)])
    d_sun        = r_sun - pos;  a_sun_arr[i]  = G*M_Sun  *(d_sun /np.linalg.norm(d_sun)**3  - r_sun /np.linalg.norm(r_sun)**3)
    d_moon       = r_moon- pos;  a_moon_arr[i] = G*M_Moon *(d_moon/np.linalg.norm(d_moon)**3 - r_moon/np.linalg.norm(r_moon)**3)
    sc_sun       = pos - r_sun;  P_sc = P_solar*(AU/np.linalg.norm(sc_sun))**2
    a_srp_arr[i] = (C_R*P_sc*A_sc/m_sc)*(sc_sun/np.linalg.norm(sc_sun))

labels = ['J2', 'Sun 3rd-body', 'Moon 3rd-body', 'SRP']
arrays = [a_J2_arr, a_sun_arr, a_moon_arr, a_srp_arr]
plt.figure(figsize=(11, 5))
for arr, lbl in zip(arrays, labels):
    plt.semilogy(t_hours, np.linalg.norm(arr, axis=1), label=lbl)
plt.xlabel("Time (hours)")
plt.ylabel("Acceleration magnitude (m/s²)")
plt.title("Perturbation Contributions Along Orbit")
plt.legend()
plt.grid(True, which='both', ls='--', alpha=0.5)
plt.tight_layout()
plt.savefig("perturbation_breakdown.png", dpi=150)
plt.show()


In [ ]:
# Export trajectory and constraint force history
df = pd.DataFrame({
    "time_s"          : t,
    "r1_x": sim.r1[:,0], "r1_y": sim.r1[:,1], "r1_z": sim.r1[:,2],
    "v1_x": sim.v1[:,0], "v1_y": sim.v1[:,1], "v1_z": sim.v1[:,2],
    "r2_x": sim.r2[:,0], "r2_y": sim.r2[:,1], "r2_z": sim.r2[:,2],
    "v2_x": sim.v2[:,0], "v2_y": sim.v2[:,1], "v2_z": sim.v2[:,2],
    "Fx_N": sim.f2_constraint[:,0],
    "Fy_N": sim.f2_constraint[:,1],
    "Fz_N": sim.f2_constraint[:,2],
})
df.to_csv("simulation_results_test.csv", index=False)
print("Results saved to simulation_results.csv")
